# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the [FAIR<sup>2</sup> dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/) using the `mlcroissant` library, which provides easy programmatic access to Croissant-compliant datasets.

### Dataset Source
The dataset is described by a Croissant schema available at the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Printing some basic information about the dataset
print(f"Name: {dataset.metadata.name}\n")
print(f"Description: {dataset.metadata.description}\n")
print(f"Version: {dataset.metadata.version}")
print(f"License: {dataset.metadata.license}")
print(f"Published: {dataset.metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All entities are referenced by their `@id` fields for traceability and reproducibility.

Let's extract an overview of record sets and their field IDs:

In [ ]:
# List all record sets defined in the croissant schema
record_sets = dataset.metadata.recordSet
if not record_sets or len(record_sets) == 0:
    print("No record sets found in the top-level metadata. Attempting to infer from distribution...")
    # fallback: sometimes recordSet can be defined under 'distribution' or elsewhere
    record_sets = []
    for dist in dataset.metadata.distribution:
        if hasattr(dist, 'recordSet') and dist.recordSet:
            record_sets.extend(dist.recordSet)
if not record_sets or len(record_sets) == 0:
    # fallback: try to list using dataset interface
    print("Could not find recordSet metadata. Querying all available record set IDs from dataset:")
    record_sets = dataset.record_sets

# Now record_sets is a list of record set @ids
print(f"Found {len(record_sets)} record set(s):")
for rs_id in record_sets:
    print(f"- RecordSet @id: {rs_id}")
    rs_meta = dataset.metadata.get_by_id(rs_id)
    if rs_meta:
        if hasattr(rs_meta, 'field'):
            print("    Fields (by @id):")
            for f in rs_meta.field:
                print(f"      - {f}")
        else:
            print("    No field metadata found.")
    else:
        print("    (Could not lookup record set metadata by @id)")

## 3. Data Extraction
Load data from a specific record set into a `DataFrame` for analysis. Use the `@id` for the desired record set and the field IDs from above.

Let's demonstrate loading the primary record set. In this dataset, the main record set appears to be:

`cr:recordSet/proteins`

*This @id may need to be adjusted if a different record set appears in the overview. Replace `cr:recordSet/proteins` with the actual record set @id of interest.*

In [ ]:
# Specify the record set you want to load (update this @id based on overview above)
RECORD_SET_ID = "cr:recordSet/proteins"  # ← update if necessary, e.g., record_sets[0]

# Collect all record sets into DataFrames
dataframes = {}
for record_set in record_sets:
    print(f"Loading records for {record_set} ...")
    records = list(dataset.records(record_set=record_set))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set] = df
        print(f"Loaded {len(df)} records. Columns: {df.columns.tolist()}")
    else:
        print("No records found for this record set.")

# Show the first few records for a selected record set
if RECORD_SET_ID in dataframes:
    print(dataframes[RECORD_SET_ID].head())
else:
    print(f"Record set {RECORD_SET_ID} not loaded or has no records.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data. All field access is by `@id` (column name as loaded, which typically matches the field @id).

Suppose (based on the record set's fields) the numeric field of interest is the peptide count, whose field `@id` is `cr:field/peptide_count`, and we want to group by the sample type (`cr:field/sample_type`). Update these `@id`s as needed according to your dataset overview.

In [ ]:
# Select the numeric field for analysis and a grouping field
numeric_field_id = "cr:field/peptide_count"   # e.g., peptide count
group_field_id = "cr:field/sample_type"       # e.g., sample type

df = dataframes.get(RECORD_SET_ID)
if df is not None and numeric_field_id in df.columns:
    # Filter for peptide_count > 10
    threshold = 10
    filtered = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered[[numeric_field_id, group_field_id]].head() if group_field_id in filtered.columns else filtered[[numeric_field_id]].head())
    
    # Normalize the numeric field
    filtered[f"{numeric_field_id}_normalized"] = (filtered[numeric_field_id] - filtered[numeric_field_id].mean()) / filtered[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} (first 5 values):")
    print(filtered[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    
    # Grouped mean by sample_type
    if group_field_id in filtered.columns:
        grouped = filtered.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:\n{grouped.head()}")
else:
    print(f"Numeric field {numeric_field_id} not found in {RECORD_SET_ID} columns.")

## 5. Visualization
Visualize the distribution of the numeric field (e.g., `peptide_count`) and the grouped means by sample type (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id], bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id in df.columns:
        grouped = df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        plt.figure(figsize=(8, 5))
        sns.barplot(data=grouped, x=group_field_id, y=numeric_field_id)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("Cannot create visualizations: required fields not present.")

## 6. Conclusion
- We loaded the dataset metadata and record sets using `mlcroissant`.
- We explored the structure and identified available record sets and fields using their unique `@id` fields.
- We extracted one record set into a DataFrame, performed filtering, normalization, and grouping on a numeric field, and visualized key relationships.
- This workflow can be extended to other record sets or fields by updating the `@id` references accordingly.